In [16]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, roc_auc_score
from collections import Counter
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from collections import Counter

In [17]:
# df_original = pd.read_csv('Children Recode_final.csv')
# df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis=1)
# df = df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1)
# df.head()

# X = df.drop(columns=['Malnurished'])
# y = df['Malnurished']

# # Train-test Split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state= 12)

# # Columns to scale
# columns_to_scale = ['Child_age', 'Age_first_sex', 'BMI', 'Mother_age_current', 'Mother_age_at_first_birth']
# scaler = StandardScaler()

# # Make copies of training and test sets
# X_train_scaled = X_train.copy()
# X_test_scaled = X_test.copy()

# # Scale only selected columns
# X_train_scaled[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
# X_test_scaled[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

# # Apply SMOTE
# sm = SMOTE(random_state=42)
# X_train_sm, y_train_sm = sm.fit_resample(X_train_scaled, y_train)

# print("Before SMOTE:", Counter(y_train))
# print("After SMOTE:", Counter(y_train_sm))

**ANN Model**

In [18]:
# # Build the ANN model
# model = Sequential([
#     Dense(64, input_dim=X_train_sm.shape[1], activation='relu'),
#     Dropout(0.3),
#     Dense(32, activation='relu'),
#     Dense(1, activation='sigmoid')  # Binary output
# ])

# # Compile the model
# model.compile(
#     optimizer=Adam(learning_rate=0.001),
#     loss='binary_crossentropy',
#     metrics=['Recall', 'Precision']
# )

# # Train the model
# history = model.fit(
#     X_train_sm, y_train_sm,
#     epochs=50,
#     batch_size=32,
#     validation_split=0.2,
#     verbose=1
# )


In [19]:

# # Evaluate on test set
# y_pred_prob = model.predict(X_test_scaled).flatten()
# y_pred = (y_pred_prob >= 0.5).astype(int)

# print("Classification Report on Test Set:")
# print(classification_report(y_test, y_pred))

In [20]:
# Load Data
df_original = pd.read_csv('df.csv')
df_original['Malnurished'] = df_original[['Underweight', 'Stunting', 'Wasting']].max(axis=1)
df = df_original.drop(['Underweight', 'Stunting', 'Wasting'], axis = 1)

# Train-test Split
X = df.drop(columns=['Malnurished'])
y = df['Malnurished']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, stratify = y, random_state = 42)

# Standard Scaler
columns_to_scale = ['Child_age', 'BMI', 'Mother_age_current']
scaler = StandardScaler()
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

In [21]:
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from sklearn.utils import class_weight


In [23]:
# Step 5: Build the Artificial Neural Network model
model = Sequential()
# Input layer with 32 neurons; adjust the number of neurons based on your dataset's complexity
model.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
# You can add a dropout layer to reduce overfitting
model.add(Dropout(0.2))
# A hidden layer with 16 neurons
model.add(Dense(16, activation='relu'))
# Output layer with a single neuron and sigmoid activation for binary classification
model.add(Dense(1, activation='sigmoid'))

# Step 6: Compile the model
# Use binary_crossentropy as the loss function for binary classification
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['recall'])

# Step 7: Train the model
# Use early stopping to halt training once the validation loss stops decreasing
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,  # Use 20% of the training set for validation
    epochs=100,            # You can increase epochs if your dataset is larger
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# Step 8: Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

# Make predictions on the test set
y_pred = model.predict(X_test).ravel()
# Convert probabilities to binary outcomes (using 0.5 as the threshold)
y_pred_binary = (y_pred > 0.5).astype(int)

# Display evaluation metrics
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_binary))
print("Classification Report:")
print(classification_report(y_test, y_pred_binary))

Epoch 1/100


/home/codespace/.python/current/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.6612 - recall: 0.1807 - val_loss: 0.8733 - val_recall: 0.1004
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6541 - recall: 0.1117 - val_loss: 0.8849 - val_recall: 0.0919
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6421 - recall: 0.1207 - val_loss: 0.8874 - val_recall: 0.0833
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6371 - recall: 0.1189 - val_loss: 0.8814 - val_recall: 0.1816
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6420 - recall: 0.1869 - val_loss: 0.9101 - val_recall: 0.1389
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6199 - recall: 0.1847 - val_loss: 0.8404 - val_recall: 0.2671
Epoch 7/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6192 - recall: 0.1645 - val_loss: 0.8387 - val_recall: 0.2778
Epoch 8/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6348 - recall: 0.2674 - val_loss: 0.8690 - val_recall: 0.2607
Epoch 9/100


In [24]:
from sklearn.metrics import average_precision_score
print(f'Average Precision: {average_precision_score(y_test, y_pred)}')

Average Precision: 0.41546395847414863
